# Felix Mumo - Assignment 10
## Deep Learning: Computer Vision (MNIST)

In this assignment I build an Artificial Neural Network with TensorFlow/Keras to classify handwritten digits (0 to 9) from the MNIST dataset. I preprocess the images, build and train the model, evaluate it on the test set, and save it in the Keras format.

### 1. Import libraries

In [ ]:
import datetime
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import TensorBoard
from sklearn.metrics import confusion_matrix, classification_report

tf.keras.utils.set_random_seed(42)
print("TensorFlow version:", tf.__version__)

### 2. Load the MNIST dataset

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

### 3. Visualize random images

In [ ]:
random_idx = np.random.choice(len(X_train), 9, replace=False)

plt.figure(figsize=(7, 7))
for i, idx in enumerate(random_idx):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_train[idx], cmap="gray")
    plt.title(f"Label: {y_train[idx]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

I also checked how many images there are for each digit. The classes are fairly balanced.

In [ ]:
digits, counts = np.unique(y_train, return_counts=True)
plt.bar(digits, counts)
plt.xticks(digits)
plt.xlabel("Digit")
plt.ylabel("Number of images")
plt.title("Images per digit (training set)")
plt.show()

### 4. Preprocessing
Pixel values go from 0 to 255, so I divide by 255 to get values between 0 and 1. The labels are one-hot encoded because the output layer has 10 neurons.

In [ ]:
print("Pixel range before:", X_train.min(), "to", X_train.max())

X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

print("Pixel range after:", X_train.min(), "to", X_train.max())

In [ ]:
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

print("Original label:", y_train[0])
print("One-hot label:", y_train_cat[0])

In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train_cat shape:", y_train_cat.shape)
print("y_test_cat shape:", y_test_cat.shape)

### 5. Build the model
- Flatten turns each 28x28 image into 784 values (input layer)
- Two hidden Dense layers with 128 and 64 neurons and ReLU activation
- Dropout of 0.3 after each hidden layer to reduce overfitting
- Output layer with 10 neurons and softmax, one for each digit

In [ ]:
model = Sequential([
    Input(shape=(28, 28)),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(10, activation="softmax")
])

model.summary()

### 6. Compile the model

In [ ]:
model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

### 7. Train the model with a TensorBoard callback

In [ ]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

history = model.fit(X_train, y_train_cat,
                    epochs=10,
                    batch_size=128,
                    validation_split=0.1,
                    callbacks=[tensorboard_callback])

Viewing the training logs in TensorBoard:

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

### 8. Plot training and validation accuracy and loss

In [ ]:
epochs = range(1, len(history.history["accuracy"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, history.history["accuracy"], marker="o", label="Training accuracy")
plt.plot(epochs, history.history["val_accuracy"], marker="o", label="Validation accuracy")
plt.title("Accuracy per epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history.history["loss"], marker="o", label="Training loss")
plt.plot(epochs, history.history["val_loss"], marker="o", label="Validation loss")
plt.title("Loss per epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

### 9. Evaluate on the test set

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)

print("Test loss:", round(test_loss, 4))
print(f"Test accuracy: {test_accuracy * 100:.2f}%")

### 10. Predictions

In [ ]:
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("First 10 predictions:", y_pred[:10])
print("First 10 true labels:", y_test[:10])

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[i], cmap="gray")
    color = "green" if y_pred[i] == y_test[i] else "red"
    plt.title(f"Pred: {y_pred[i]} / True: {y_test[i]}", color=color, fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

### 11. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix")
plt.show()

### 12. Classification report

In [ ]:
print(classification_report(y_test, y_pred, digits=4))

Some of the images the model got wrong:

In [ ]:
wrong = np.where(y_pred != y_test)[0]
print("Number of wrong predictions:", len(wrong), "out of", len(y_test))

plt.figure(figsize=(10, 4))
for i, idx in enumerate(wrong[:10]):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[idx], cmap="gray")
    plt.title(f"Pred: {y_pred[idx]} / True: {y_test[idx]}", color="red", fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

### 13. Save the model (Keras format)

In [ ]:
model.save("mnist_ann_model.keras")
print("Model saved as mnist_ann_model.keras")

Loading the saved model to check it works:

In [ ]:
loaded_model = load_model("mnist_ann_model.keras")

loss, acc = loaded_model.evaluate(X_test, y_test_cat, verbose=0)
print(f"Loaded model test accuracy: {acc * 100:.2f}%")

sample = X_test[0].reshape(1, 28, 28)
print("Prediction for first test image:", np.argmax(loaded_model.predict(sample, verbose=0)))
print("True label:", y_test[0])

### 14. Conclusion
- I normalized the images to the range 0 to 1 and one-hot encoded the labels before training.
- The ANN has two hidden layers (128 and 64 neurons) with dropout of 0.3, and was trained for 10 epochs.
- The training and validation curves stay close together, so the dropout layers helped prevent overfitting.
- The model reached a test accuracy of about 98% (see the output in section 9).
- From the confusion matrix, most mistakes happen between digits that look alike, such as 4 and 9, 3 and 5, or 7 and 2.
- The model was saved in the native Keras format (.keras) and loaded again successfully.